=================================================
### Práctica 4b — Detección y Seguimiento con YOLOv8
=================================================

##### Este cuaderno realiza:
1. Detección y seguimiento de personas y vehículos con **YOLOv8**.
2. Detección de matrículas con un modelo propio entrenado.
3. Lectura de matrículas
4. Generación de un **CSV con detecciones y tracking**.


### Carga de modelos y configuración inicial

In [178]:
from ultralytics import YOLO
import cv2, csv
from collections import defaultdict, deque
import pandas as pd

# Archivos de entrada y salida
video_input = "C0142.mp4"
video_output = "p4_output.mp4"
csv_output   = "p4_results.csv"

# Modelos YOLO
general_model = YOLO('yolo11n.pt').to('cuda')  # Detección general (GPU)
plate_model   = YOLO('yolo_runs/plates_detection/weights/best.pt').to('cuda')  # Detección de matrículas (GPU)

# Clases relevantes
classNames = ["person", "bicycle", "car", "motorbike", "", "bus"]
general_classes = [0,1,2,3,4,5]

# Diccionario de colores (BGR)
class_colors = {
    "person": (255, 255, 0),  # cian
    "bicycle": (0, 255, 0),
    "car": (0, 0, 255),
    "motorbike": (255, 0, 0),
    "bus": (0, 165, 255),
    "truck": (128, 0, 128)
}
plate_color = (0, 255, 255)  # amarillo

# Para que cada objeto se cuente UNA vez:
counted_ids = {k: set() for k in ["person","bicycle","car","motorbike","bus","truck","plate"]}

# Para mantener la clase "canónica" del track_id (la primera clase que tuvo)
track_classes = {}   # track_id -> clase_asignada

# Historial de centros (para calcular flujo)
track_history = defaultdict(list)    # track_id -> list of (cx,cy)

# Contadores en tiempo real (únicos)
total_by_classes = defaultdict(int)  # por clase (person, car, ...)
exit_count = {"arriba": defaultdict(int), "abajo": defaultdict(int)}

# IDs presentes en el frame anterior (para detectar salidas)
prev_ids = set()
# Para no contar dos veces la salida:
counted_exit_ids = set()

### Preparación del vídeo

In [179]:
# Carga del vídeo
vid = cv2.VideoCapture(video_input)
width  = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = vid.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_vid = cv2.VideoWriter(video_output, fourcc, fps, (width, height))

# Historial de tracking
track_history = defaultdict(lambda: deque(maxlen=5))

# Umbrales de salida vertical (5% de margen)
top_exit_threshold = int(height * 0.05)
bot_exit_threshold = height - top_exit_threshold

### Funciones auxiliares para el CSV

In [180]:
def write_csv_header(path):
    with open(path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["frame","tipo_objeto","confianza","track_id","x1","y1","x2","y2",
                         "plate","plate_conf","mx1","my1","mx2","my2"])

def append_detection(path, frame, obj_class, conf, track_id,
                     x1, y1, x2, y2, plate, plate_conf, mx1, my1, mx2, my2):
    with open(path, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([frame,obj_class,conf,track_id,x1,y1,x2,y2,
                         plate,plate_conf,mx1,my1,mx2,my2])

write_csv_header(csv_output)

### Función de lectura con OCR tesseract

In [181]:
import pytesseract
import numpy as np

def readPlateOCR (img):
    #cv2.imwrite("testimg.jpg",img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    #cv2.imwrite("testtresh.jpg",thresh)

    items = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = items[0] if len(items) == 2 else items[1]

    img_contour = img.copy()
    
    for i in range(len(contours)):
        area = cv2.contourArea(contours[i])
        if 2 < area < 10000:
            cv2.drawContours(img_contour, contours, i, (0, 0, 255), 1)
    
    detected = ""
    for c in reversed(contours):
        x, y, w, h = cv2.boundingRect(c)
        ratio = h/w
        area = cv2.contourArea(c)
        base = np.ones(thresh.shape, dtype=np.uint8)
        if ratio > 0.9 and 2 < area < 10000:
            base[y:y+h, x:x+w] = thresh[y:y+h, x:x+w]
            segment = cv2.bitwise_not(base)
            custom_config = r'-l spa --oem 3 --psm 10 '
            c = pytesseract.image_to_string(segment, config=custom_config)
            c = c.replace("\n","")
            detected = detected + c

    if detected == "":
        return "None"
    else:
        return detected


### Función de lectura con SmolVLM

Requiere paquetes 'transformers' y 'flash-attn'

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")
# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Read the text in the image and answer only with 'Text: <the text>' even if there is no text:"}
        ]
    },
]
if 'model' not in locals():
    model = AutoModelForVision2Seq.from_pretrained("HuggingFaceTB/SmolVLM-Instruct",
                                                torch_dtype=torch.bfloat16).to(DEVICE)
                                                

In [183]:
def readPlateVLM (img):
    prompt = processor.apply_chat_template(messages)
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    outputs = model.generate(**inputs, max_new_tokens=6)
    decoded = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    
    plate_text = decoded.split("Text:")
    if len(plate_text) == 3:
        print (plate_text)
        plate_text = plate_text[2].strip()
    else:
        return "None"
        
    if len(plate_text) == 0:
        return "None"
    
    if plate_text[-1] == ".":
        plate_text = plate_text[:-1]
    return plate_text

### Procesamiento frame a frame con detección, seguimiento y anonimización

In [184]:
frame_n = 0

while True:
    ret, frame = vid.read()
    if not ret:
        break
    frame_n += 1

    results = general_model.track(frame, persist=True, classes=general_classes)

    current_ids = set()  # ids detectados este frame

    # results es un iterable (stream) con un elemento por imagen - aquí uno solo
    for r in results:
        for box in r.boxes:
            # extraer info
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            # convertir id
            track_id = int(box.id[0]) if box.id is not None else -1
            if track_id == -1:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            # si el índice de clase no está en nuestro mapping, omitimos
            if cls >= len(classNames):
                continue
            detected_class = classNames[cls] if cls < len(classNames) else None
            if detected_class is None:
                continue

            current_ids.add(track_id)

            # --- Mantener clase consistente: si no tenía clase asignada, la fijamos ahora ---
            if track_id not in track_classes:
                track_classes[track_id] = detected_class
                # contar la aparición **única** del objeto en su clase (solo la primera vez)
                if track_id not in counted_ids.get(detected_class, set()):
                    counted_ids.setdefault(detected_class, set()).add(track_id)
                    total_by_classes[detected_class] += 1

            # usamos la clase asignada (no la detectada para evitar que cambie)
            clase = track_classes.get(track_id, detected_class)

            # --- Actualizar historial de centros (para flujo vertical) ---
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            track_history[track_id].append((cx, cy))

            # --- Detectar placa ---
            plate_conf, mx1, my1, mx2, my2 = "", "", "", "", ""
            has_plate = False
            plate_text = ""
            if clase not in ["person", "bicycle"]:
                vehicle_crop = frame[y1:y2, x1:x2]
                if vehicle_crop.size > 0 and vehicle_crop.shape[0] > 2 and vehicle_crop.shape[1] > 2:
                    plate_results = plate_model(vehicle_crop)
                    best_conf = 0.0
                    best_box = None
                    for p in plate_results:
                        for pb in p.boxes:
                            conf_plate = float(pb.conf[0])
                            if conf_plate > best_conf:
                                best_conf = conf_plate
                                best_box = pb
                    if best_box is not None and best_conf > 0.15:
                        has_plate = True
                        x1p, y1p, x2p, y2p = map(int, best_box.xyxy[0])
                        plate_conf = best_conf
                        mx1, my1, mx2, my2 = x1p, y1p, x2p, y2p
                        plate_region = vehicle_crop[y1p:y2p, x1p:x2p]
                        #plate_text = readPlateOCR(plate_region)
                        plate_text = readPlateVLM(plate_region)

                        # dibujar rectángulo absoluto de placa en frame
                        ax1, ay1 = x1 + mx1, y1 + my1
                        ax2, ay2 = x1 + mx2, y1 + my2
                        cv2.rectangle(frame, (ax1, ay1), (ax2, ay2), plate_color, 2)

                        # contabilizar "car plates" único por track_id
                        if track_id not in counted_ids["plate"]:
                            counted_ids["plate"].add(track_id)
                            total_by_classes["plate"] += 1

            plate_text = plate_text if has_plate else "No"

            # --- Dibujar bbox principal con color por clase ---
            color = class_colors.get(clase, (255,255,255))
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, f"[{track_id}] {clase} {conf:.2f}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # --- Guardar fila en CSV (sin flujo) ---
            append_detection(csv_output, frame_n, clase, conf, track_id, x1,y1,x2,y2, plate_text, plate_conf, mx1,my1,mx2,my2)

    # ----------------------------
    # Detectar IDs que han desaparecido (salida) comparando prev_ids y current_ids
    # ----------------------------
    disappeared = prev_ids - current_ids
    for tid in disappeared:
        # si ya fue contado como salida, saltar
        if tid in counted_exit_ids:
            continue
        # si no tenemos historial, ignorar
        if tid not in track_history or tid not in track_classes:
            # limpiar si queda basura
            track_history.pop(tid, None)
            track_classes.pop(tid, None)
            continue

        # Obtener primera y última cy para decidir dirección
        centers = track_history[tid]
        if len(centers) >= 5:
            cy_first = centers[0][1]
            cy_last  = centers[-1][1]
            clase = track_classes.get(tid, "unknown")
            # Preferimos analizar si salió por borde (top/bottom) usando última posición
            if centers[-1][1] <= top_exit_threshold:
                direction = "arriba"
            elif centers[-1][1] >= bot_exit_threshold:
                direction = "abajo"
            else:
                # si no ha salido por el borde, usar comparación first/last
                if cy_last < cy_first:
                    direction = "arriba"
                elif cy_last > cy_first:
                    direction = "abajo"
                else:
                    direction = "estatico"
            if direction in ["arriba", "abajo"]:
                exit_count[direction][clase] += 1
            counted_exit_ids.add(tid)

        # limpiar datos del track
        track_history.pop(tid, None)
        track_classes.pop(tid, None)

    prev_ids = current_ids.copy()

    # ----------------------------
    # Mostrar contadores frame-a-frame en el frame
    # ----------------------------
    # Dibujar cuadro semi-opa para visibilidad
    panel_h = 150
    #cv2.rectangle(frame, (5,5), (270, 5+panel_h), (30,30,30), -1)  # fondo oscuro
    y0 = 25
    delta = 22
    # Totales por clase (únicos)
    for i, clsname in enumerate(["person","car","motorbike","bus","bicycle","plate"]):
        val = total_by_classes.get(clsname, 0)
        txt = f"{clsname}: {val}"
        cv2.putText(frame, txt, (10, y0 + i*delta), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (240,240,240), 1, cv2.LINE_AA)

    # Flujos registrados (salidas)
    y_flow = 10 + panel_h
    #cv2.rectangle(frame, (5, y_flow), (270, y_flow+90), (30,30,30), -1)
    y0f = y_flow + 25
    cv2.putText(frame, f"Arriba:", (10, y0f), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 1)
    y0f += 22
    cnt = exit_count["arriba"]
    for k, v in cnt.items():
        cv2.putText(frame, f"{k}:{v}", (12, y0f), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 1); y0f += 18
    # abajo
    y0f = y_flow + 25
    cv2.putText(frame, f"Abajo:", (140, y0f), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 1)
    y0f += 22
    cnt2 = exit_count["abajo"]
    for k, v in cnt2.items():
        cv2.putText(frame, f"{k}:{v}", (142, y0f), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 1); y0f += 18

    # Escribir frame procesado en vídeo de salida
    out_vid.write(frame)

# Fin loop
vid.release()
out_vid.release()
print("CSV de detecciones generado:", csv_output)
print("Video de detecciones generado:", video_output)


0: 384x640 4 cars, 1 bus, 14.0ms
Speed: 2.4ms preprocess, 14.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 352x416 (no detections), 10.8ms
Speed: 1.3ms preprocess, 10.8ms inference, 0.5ms postprocess per image at shape (1, 3, 352, 416)

0: 320x416 (no detections), 11.7ms
Speed: 1.0ms preprocess, 11.7ms inference, 0.7ms postprocess per image at shape (1, 3, 320, 416)

0: 288x416 (no detections), 11.3ms
Speed: 0.9ms preprocess, 11.3ms inference, 0.7ms postprocess per image at shape (1, 3, 288, 416)

0: 288x416 (no detections), 11.0ms
Speed: 0.8ms preprocess, 11.0ms inference, 0.5ms postprocess per image at shape (1, 3, 288, 416)

0: 320x416 (no detections), 11.4ms
Speed: 0.9ms preprocess, 11.4ms inference, 0.8ms postprocess per image at shape (1, 3, 320, 416)

0: 384x640 4 cars, 1 bus, 12.7ms
Speed: 3.1ms preprocess, 12.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 352x416 1 plate, 10.6ms
Speed: 1.4ms preprocess, 10.6ms inference, 1.

KeyboardInterrupt: 